# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'ht

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [7]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [8]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2024/12/21/

In [9]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [10]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/deepseek-ai/DeepSeek-R1',
 '/Wan-AI/Wan2.1-T2V-14B',
 '/microsoft/Phi-4-multimodal-instruct',
 '/perplexity-ai/r1-1776',
 '/allenai/olmOCR-7B-0225-preview',
 '/models',
 '/spaces/Wan-AI/Wan2.1',
 '/spaces/nanotron/ultrascale-playbook',
 '/spaces/huggingface/ai-deadlines',
 '/spaces/black-forest-labs/FLUX.1-dev',
 '/spaces/lllyasviel/LuminaBrush',
 '/spaces',
 '/datasets/facebook/natural_reasoning',
 '/datasets/Congliu/Chinese-DeepSeek-R1-Distill-data-110k',
 '/datasets/SynthLabsAI/Big-Math-RL-Verified',
 '/datasets/FreedomIntelligence/medical-o1-reasoning-SFT',
 '/datasets/allenai/olmOCR-mix-0225',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel'

In [11]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'career page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [12]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [13]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Posts
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates

In [22]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nPosts\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-R1\nUpdated\n6 days ago\n•\n4.55M\n•\n10.6k\nWan-AI/Wan2.1-T2V-14B\nUpdated\n3 days ago\n•\n135k\n•\n629\nmicrosoft/Phi-4-multimodal-instruct\nUpdated\n1 day ago\n•\n15.1k\n•\n607\nperplexity-ai/r1-1776\nUpdated\n3 days ago\n•\n33.5k\n•\n1.92k\nallenai/olmOCR-7B-0225-preview\nUpdated\n5 days ago\n•\n27k\n•\n332\nBrowse 1M+ models\nSpaces\nRunning\n641\n641\nWan2.1\n💻\nWan:

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


```markdown
# Welcome to Hugging Face 🌟

## "The AI community building the future... one hug at a time!" 🤗

---

### About Us

At Hugging Face, we're not just hugging the world - we're hugging algorithms, datasets, and a vibrant community of AI enthusiasts! We're the friendly bunch shaping the future of machine learning, and we've got over **1 million models**, **400k applications**, and **250k datasets** all wrapped in a big, warm bear hug! 

---

### What We Offer

- **Models Galore**: Tired of the same old boring models? Come check out our trendy, up-to-date models that have more personality than your average AI! 🥳 
- **Datasets Galore**: Your data deserves better! Browse through our diverse array of datasets that are more organized than your sock drawer. 🧦📊
- **Spaces Galore**: We’ve got 641 running spaces ready for you to flex your ML muscles and keep your models comfy! 💻🛋️

---

### Our Customers

Join the club of **50,000+ organizations** using Hugging Face, including household names like Amazon, Google, and Microsoft. 🎉 We help companies create better AI; think of us as the friendly neighborhood superhero of machine learning!

---

### Company Culture

At Hugging Face, we prioritize collaboration! 🎈 The vibe is less 'corporate suit' and more 'cool, laid-back genius'. We believe that sharing ideas is essential, just like sharing snacks during those late-night coding sessions. 🍕

We’re a community that encourages innovation: Bring your craziest AI ideas and see how many talented folks can help you turn them into reality. The only thing we don’t share is bad vibes!

---

### Careers at Hugging Face

Looking to join us? Check out our **hired hearts!** 💖 We’re on the lookout for passionate individuals who can help us democratize machine learning! Think you're up for the challenge? We also promise to provide ample office hugs (as long as you're cool with that)! 

---

### Pricing? You Ask, We Hug!

- **Compute**: Starting at just $0.60 per hour for our GPU magic! 

- **Enterprise Solutions**: You can get started for just $20 per user/month. (*because who doesn't love a budget-friendly hug?*) 

Jump in and explore our plans that make powerful AI accessible to all without breaking the bank—because hugs shouldn’t come with a hefty price tag! 

---

## Why Hugging Face?

Because we’re not just here to give you a service; we’re here to build a family. At Hugging Face, every hug is a metaphorical algorithm, and every giggle is a breakthrough moment. Join us on this journey to make AI accessible, friendly, and, let’s face it, a lot more fun!

### Let’s Hug It Out! 🤗
```


## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# 🐻🐾 Welcome to Hugging Face: Where AI Gets a Cozy Embrace! 🐾🐻

## The AI Community Building the Future
Forget about rocket science; we’re diving into the world of AI where hugging machines is as natural as hugging your grandma! 

### Who Are We?
At Hugging Face, we believe in love at first byte. Our community builds models, shares datasets, and collaborates on applications that make AI blossom like never before. With over **1 million models** to choose from, your dreams of AI domination are just a click away!

### What Do We Offer?
- **Models Galore!** 🎭: From deep learning models that make even Einstein seem outdated, to generative models that create delightful chaos.
- **Datasets for Days!** 📊: Bored of the same old data? Browse through **250K+ datasets** that’ll keep your machine learning hamster wheel spinning!
- **Spaces to Explore!** 🚀: Think of it as your playground for AI applications where the only limit is your imagination.

### **Trending Models This Week**
1. `deepseek-ai/DeepSeek-R1` - Because sometimes, you just need to know what’s beneath the surface! 
2. `microsoft/Phi-4-multimodal-instruct` - For when you can't decide if you want AI instructions for cooking or building a robot army! 
3. `perplexity-ai/r1-1776` - Perfect for perplexing times!

### Our Community is HUGE!
Join over **50,000 organizations**, including giants like Google and Microsoft, who are already hugging it out with us! 

## Career Opportunities
Come work with us, and you'll be in the best company ever (pun intended). We prioritize a collaborative, inclusive culture where big ideas are born. Plus, you can finally justify all those late-night coding binges!

### Roles Available:
- **Machine Learning Wizards**: Must excel in spells involving neural networks and deep learning.
- **Data Whisperers**: Can talk to databases and datasets (you'll fit right in).
- **Creative Coders**: If your idea of painting is writing lines of code that create stunning visuals, we want you!

### Why Hugging Face?
1. **A Warm Community**: Like a group hug, minus the awkwardness!
2. **Tools that Empower**: More tools than your local toolbox!
3. **Flexibility**: Work at your pace, as long as you don’t take “work” too literally.

### Join the Hugging Face Adventure!
Ready to change the world and make it a better place, one byte at a time? Sign up today and embrace the future of AI!

**Don't be shy! Hit us up at: [huggingface.co](https://huggingface.co)**

---

*P.S. No bears were harmed in the making of our logo, but they definitely inspired it!* 🐻✨

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'community discussion page', 'url': 'https://discuss.huggingface.co'}]}


# Welcome to the **Hugging Face** Brochure! 🤗

---

### **Who Are We?**

At **Hugging Face**, we’re not just building AI; we’re crafting a cuddle-worthy community dedicated to everyone from code-crunching techies to casual AI experimenters. We like our models like we like our hugs—**big and warm**!

---

### **What Do We Do?**

You won’t believe it unless you see it! At **Hugging Face**, you can explore:

-  **1M+ Models**: It’s like a buffet but for AI! Choose from *text, image, video, audio*, or even *3D*. Just don’t forget to leave room for dessert (or your favorite dataset)!
  
-  **Spaces**: Not the pretend-personal-space kind, but real-deal areas to host and collaborate on endless public models, datasets, and applications.

-  **AI Apps**: Because why not let AI handle **that** while you kick your feet up? Explore the coolest apps in the community! 

---

### **Our Customers (aka Our Friends!)**

We play nice with **more than 50,000 organizations**. From tech giants like Microsoft, Google, and Amazon to smaller, less famous buddies, we’ve got them all. If you’re not on this list, don’t worry—we won’t judge! (But please come join us.)

> "For every AI model we train, we give it a special name... just like pets!" 🐾

---

### **Careers at Hugging Face** 🎉

Thinking about joining our squad? Here are *just* a few reasons to work with us:

- **Work with Cool Stuff**: We’re constantly pumping out new models, datasets, and applications. Who wouldn’t want to say they work for a company building the future?

- **Team Culture**: Our coffee is strong, our hugs are stronger! You’ll be surrounded by passionate individuals who share a love for AI and can appreciate a good meme.

- **Opportunities for Growth**: Be a part of something bigger than just a job—be part of the machine learning revolution!

---

### **Come On In!**

Are you ready to embrace the warmth of AI innovation and collaboration? We’re excited to *hug* your future (and models) with open arms! 

**Let’s build the future together!**

\[🤗](https://huggingface.co/) [Join us!](https://huggingface.co/join) 

### **Get in Touch!**
- **Follow us on**: [Twitter](https://twitter.com/huggingface), [LinkedIn](https://www.linkedin.com/company/huggingface/)
- **Check our resources**: [Docs](https://huggingface.co/docs)

> “AI might be what we make it, but our community is what makes us best!” 

--- 

*Disclaimer: Hugging Face takes no responsibility for uncontrollable enthusiasm when browsing our extensive models, we warned you!* 

--- 

Feel free to share this with your friends while you ponder which dataset to dive into next! 🤣

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>